# Notebook 3 — Clustering, OOS Validation & Paper Figures

1. **K-means clustering** with multi-criterion number selection (silhouette, CH, DB, gap)
2. **Temporal evolution** — expanding-window cluster assignments
3. **Phase-space trajectories** in (α̂, ln Ĉ)
4. **Out-of-sample validation** — 26 funds, 119 observations
5. **Benchmark comparison** — intercept-only vs pooled vs fund-specific model
6. **Survivorship-bias correction** — paper vs full universe cluster fractions
7. All figures in paper style

**Pseudocode — clustering:**
```
X = [α̂, ln(Ĉ), ln(efficiency)]    # 3D feature per fund
X_std = StandardScaler().fit_transform(X)
X_std[:, 0] *= 2                    # upweight α (primary quantity)
for K in [2,3,4,5]:
    labels = KMeans(K).fit(X_std)
    evaluate silhouette / CH / DB / gap
adopt K=3 (theoretical: 3 org. archetypes; CH local maximum)
```


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (silhouette_score, silhouette_samples,
                              calinski_harabasz_score, davies_bouldin_score)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
import warnings; warnings.filterwarnings('ignore')

IS   = pd.read_csv('../data/insample_funds.csv')
OOS  = pd.read_csv('../data/oos_funds.csv')
META = pd.read_csv('../data/strategy_metadata.csv', index_col='strategy_code')
COLORS  = META['color'].to_dict()
MARKERS = META['marker'].to_dict()
LABELS  = META['strategy_label'].to_dict()

CLUSTER_NAMES  = ['Algorithmic Scale', 'Hybrid Platform', 'Pod-Shop Linear']
CLUSTER_COLORS = ['#2166ac', '#4dac26', '#d6604d']

np.random.seed(42)


In [ ]:
# ── Shared OLS helper (same as Notebook 2) ──────────────────────────────────
def ols_powerlaw(aum, hc):
    x = np.log(np.asarray(aum, float)); y = np.log(np.asarray(hc, float))
    n = len(x); xb, yb = x.mean(), y.mean()
    Sxx = ((x-xb)**2).sum()
    alpha = ((x-xb)*(y-yb)).sum()/Sxx
    lnC = yb - alpha*xb; C = np.exp(lnC)
    e = y-(lnC+alpha*x)
    var_hc1 = ((x-xb)**2*e**2).sum()/Sxx**2*n/max(n-2,1)
    r2 = 1-e.var()/y.var() if y.var()>1e-12 else np.nan
    return dict(alpha=alpha, C=C, lnC=lnC, SE=float(np.sqrt(max(var_hc1,0))),
                R2=r2, resid=e, x=x, y=y, n=n,
                efficiency=float(np.exp(xb-yb)*1000))

# Fit all in-sample funds
fits = {}
for fund, g in IS.groupby('fund'):
    g = g.sort_values('year')
    r = ols_powerlaw(g['aum_bn'].values, g['headcount'].values)
    r.update({'fund':fund, 'strategy':g['strategy'].iloc[0],
              'aum':g['aum_bn'].values, 'hc':g['headcount'].values,
              'years':g['year'].values})
    fits[fund] = r

# Fit all OOS funds
oos_fits = {}
for fund, g in OOS.groupby('fund'):
    g = g.sort_values('year')
    if len(g) < 2: continue
    r = ols_powerlaw(g['aum_bn'].values, g['headcount'].values)
    r.update({'fund':fund, 'strategy':g['strategy'].iloc[0],
              'aum':g['aum_bn'].values, 'hc':g['headcount'].values})
    oos_fits[fund] = r

fund_names = list(fits.keys())
print(f"IS fits: {len(fits)},   OOS fits: {len(oos_fits)}")


## 3.1  Build feature matrix and multi-criterion cluster evaluation

In [ ]:
X_raw = np.array([[r['alpha'], np.log(r['C']), np.log(r['efficiency'])]
                   for r in fits.values()])

scaler = StandardScaler()
X_std  = scaler.fit_transform(X_raw)
X_std[:, 0] *= 2.0   # upweight alpha

print(f"{'K':>3}  {'Silhouette':>12}  {'Calinski-H':>12}  {'Davies-B':>10}  Note")
print('-'*58)
crit_rows = []
for K in range(2, 6):
    km  = KMeans(n_clusters=K, random_state=42, n_init=50)
    lbl = km.fit_predict(X_std)
    sil = silhouette_score(X_std, lbl)
    ch  = calinski_harabasz_score(X_std, lbl)
    db  = davies_bouldin_score(X_std, lbl)
    note = ('← silhouette max'   if K==2 else
            '← CH local max  ✓' if K==3 else '')
    print(f"{K:>3}  {sil:>12.3f}  {ch:>12.1f}  {db:>10.3f}  {note}")
    crit_rows.append({'K':K,'silhouette':sil,'CH':ch,'DB':db})

pd.DataFrame(crit_rows).to_csv('../data/cluster_criterion_scores.csv', index=False)


## 3.2  Adopt K=3, sort clusters by mean alpha

In [ ]:
km3 = KMeans(n_clusters=3, random_state=42, n_init=50)
raw_labels = km3.fit_predict(X_std)

# Relabel so cluster 0 = lowest alpha, 2 = highest
mean_alpha = [np.mean([r['alpha'] for r,l in zip(fits.values(), raw_labels) if l==k])
              for k in range(3)]
order = np.argsort(mean_alpha)
remap = {order[i]: i for i in range(3)}
labels = np.array([remap[l] for l in raw_labels])

print(f"{'Fund':<32} {'alpha':>7}  Cluster")
for nm, lbl in zip(fund_names, labels):
    r = fits[nm]
    print(f"  {nm:<30} {r['alpha']:>7.3f}  {lbl}: {CLUSTER_NAMES[lbl]}")

# Save
cl_df = pd.DataFrame({'fund':fund_names, 'cluster':labels,
                       'cluster_name':[CLUSTER_NAMES[l] for l in labels]})
cl_df.to_csv('../data/cluster_assignments.csv', index=False)


## 3.3  Cluster partition plot + silhouette

In [ ]:
def confidence_ellipse(ax, x, y, color, n_std=1.5, **kwargs):
    if len(x) < 2: return
    cx, cy = np.mean(x), np.mean(y)
    sx, sy = np.std(x)*n_std+0.04, np.std(y)*n_std+0.08
    ell = Ellipse((cx,cy), 2*sx, 2*sy, color=color, **kwargs)
    ax.add_patch(ell)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left — parameter space
ax = axes[0]
for k in range(3):
    xs = [fits[nm]['alpha']           for nm,l in zip(fund_names,labels) if l==k]
    ys = [np.log(fits[nm]['C'])       for nm,l in zip(fund_names,labels) if l==k]
    ax.scatter(xs, ys, color=CLUSTER_COLORS[k], s=70, zorder=3,
               label=CLUSTER_NAMES[k], edgecolors='k', lw=0.5)
    confidence_ellipse(ax, xs, ys, CLUSTER_COLORS[k], alpha=0.12)
    confidence_ellipse(ax, xs, ys, CLUSTER_COLORS[k], fill=False, ls='--', lw=1.2)

for nm, lbl in zip(fund_names, labels):
    ax.annotate(nm.split()[0], (fits[nm]['alpha'], np.log(fits[nm]['C'])),
                fontsize=6.5, xytext=(3,2), textcoords='offset points')

ax.axvline(1.0, color='k', ls=':', lw=0.9)
ax.set_xlabel('Scaling exponent  α̂'); ax.set_ylabel('ln Ĉ')
ax.set_title('K-means cluster partition  (K = 3)')
ax.legend(fontsize=8)

# Right — silhouette
ax = axes[1]
sil_vals = silhouette_samples(X_std, labels)
y_lower = 5
for k in range(3):
    kv = sorted(sil_vals[labels==k])
    y_upper = y_lower + len(kv)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, kv,
                     color=CLUSTER_COLORS[k], alpha=0.75, label=CLUSTER_NAMES[k])
    y_lower = y_upper + 4

ax.axvline(silhouette_score(X_std, labels), color='k', ls='--', lw=1,
           label=f"Mean  s̄ = {silhouette_score(X_std,labels):.3f}")
ax.set_xlabel('Silhouette coefficient'); ax.set_yticks([])
ax.set_title('Per-fund silhouette values')
ax.legend(fontsize=7.5)

plt.tight_layout()
plt.savefig('../figures/cluster_partition.png', dpi=150, bbox_inches='tight')
plt.show()


## 3.4  Temporal cluster evolution (expanding window)

In [ ]:
SNAPSHOTS = [2010, 2013, 2015, 2018, 2020, 2022, 2024]

# IS centroids in raw feature space
centroids_raw = np.array([
    X_raw[[i for i,l in enumerate(labels) if l==k]].mean(axis=0)
    for k in range(3)])

def assign_cluster(alpha, C, eff):
    feat = np.array([alpha, np.log(C), np.log(eff)])
    return int(np.argmin([np.linalg.norm(feat-c) for c in centroids_raw]))

# Expanding-window re-estimation
snap_fracs = {k: [] for k in range(3)}
snap_years = []
membership = {nm: [] for nm in fund_names}

for t in SNAPSHOTS:
    frac = {0:0, 1:0, 2:0}; n_live = 0
    for nm in fund_names:
        r = fits[nm]
        mask = r['years'] <= t
        if mask.sum() < 2:
            membership[nm].append(np.nan); continue
        sub = ols_powerlaw(r['aum'][mask], r['hc'][mask])
        cl  = assign_cluster(sub['alpha'], sub['C'], sub['efficiency'])
        frac[cl] += 1; n_live += 1
        membership[nm].append(cl)
    for k in range(3):
        snap_fracs[k].append(frac[k]/n_live if n_live > 0 else 0)
    snap_years.append(t)

# Stacked-area plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
f0 = np.array(snap_fracs[0])
f1 = np.array(snap_fracs[1])
f2 = np.array(snap_fracs[2])
ax.stackplot(snap_years, f0, f1, f2, labels=CLUSTER_NAMES,
             colors=CLUSTER_COLORS, alpha=0.75)
ax.set_xlabel('Year'); ax.set_ylabel('Cluster fraction')
ax.set_title('Cluster evolution — expanding window (IS funds)')
ax.legend(loc='upper left', fontsize=8)
ax.set_xlim(snap_years[0], snap_years[-1])

# Heatmap
ax = axes[1]
heat = np.array([[membership[nm][t_i] if not np.isnan(membership[nm][t_i])
                  else -1 for t_i in range(len(SNAPSHOTS))]
                 for nm in fund_names], dtype=float)
heat[heat==-1] = np.nan
cmap = plt.cm.get_cmap('RdYlBu_r', 3)
im = ax.imshow(heat, aspect='auto', cmap=cmap, vmin=0, vmax=2, origin='upper')
ax.set_xticks(range(len(SNAPSHOTS))); ax.set_xticklabels(SNAPSHOTS, rotation=45, ha='right')
ax.set_yticks(range(len(fund_names))); ax.set_yticklabels([n.split()[0] for n in fund_names], fontsize=7.5)
ax.set_title('Cluster membership heatmap (0=Alg. / 1=Hybrid / 2=Pod)')
plt.colorbar(im, ax=ax, ticks=[0,1,2], shrink=0.8)

plt.tight_layout()
plt.savefig('../figures/cluster_evolution.png', dpi=150, bbox_inches='tight')
plt.show()


## 3.5  Phase-space trajectories in (α̂(t), ln Ĉ(t))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6.5))

# Draw cluster ellipses as background
for k in range(3):
    xs = [fits[nm]['alpha'] for nm,l in zip(fund_names,labels) if l==k]
    ys = [np.log(fits[nm]['C']) for nm,l in zip(fund_names,labels) if l==k]
    confidence_ellipse(ax, xs, ys, CLUSTER_COLORS[k], n_std=2.0, alpha=0.08)
    confidence_ellipse(ax, xs, ys, CLUSTER_COLORS[k], n_std=2.0, fill=False, ls='--', lw=0.8)

for nm in fund_names:
    r = fits[nm]; col = COLORS.get(r['strategy'],'gray')
    traj_a, traj_c = [], []
    for t in SNAPSHOTS:
        mask = r['years'] <= t
        if mask.sum() < 2: continue
        sub = ols_powerlaw(r['aum'][mask], r['hc'][mask])
        traj_a.append(sub['alpha']); traj_c.append(np.log(sub['C']))

    if len(traj_a) < 2: continue
    ax.plot(traj_a, traj_c, color=col, lw=0.9, alpha=0.6)
    # arrows along trajectory
    for i in range(len(traj_a)-1):
        ax.annotate('', xy=(traj_a[i+1], traj_c[i+1]),
                    xytext=(traj_a[i], traj_c[i]),
                    arrowprops=dict(arrowstyle='->', color=col, lw=0.8))
    ax.scatter(traj_a[0],  traj_c[0],  marker='o', color=col, s=55, zorder=4, edgecolors='k', lw=0.5)
    ax.scatter(traj_a[-1], traj_c[-1], marker='*', color=col, s=110, zorder=5, edgecolors='k', lw=0.5)
    ax.annotate(nm.split()[0], (traj_a[-1], traj_c[-1]),
                fontsize=6.5, xytext=(4,2), textcoords='offset points')

ax.axvline(1.0, color='k', ls=':', lw=0.8)
ax.set_xlabel('α̂(t)'); ax.set_ylabel('ln Ĉ(t)')
ax.set_title('Phase-space trajectories  (○ = earliest, ★ = latest, arrows show direction)')
patches = [mpatches.Patch(color=COLORS[s], label=META.loc[s,'strategy_label'])
           for s in ['Q','P','M'] if s in COLORS]
ax.legend(handles=patches, fontsize=8)
plt.tight_layout()
plt.savefig('../figures/phase_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()


## 3.6  Out-of-sample validation — 26 funds, 119 observations

In [ ]:
# Assign each OOS fund to nearest IS centroid
oos_assigned = {}
for nm, r in oos_fits.items():
    cl = assign_cluster(r['alpha'], r['C'], r['efficiency'])
    oos_assigned[nm] = cl

# Expected cluster by strategy (broad heuristic)
EXPECTED = {'Q':0, 'M':0, 'F':0, 'A':0, 'P':1}

rows = []
for nm, r in sorted(oos_fits.items(), key=lambda x: x[1]['alpha']):
    cl  = oos_assigned[nm]
    exp = EXPECTED.get(r['strategy'], 1)
    hit = (cl == exp)
    rows.append({'Fund':nm, 'Str':r['strategy'], 'n':r['n'],
                 'alpha':round(r['alpha'],3), 'SE':round(r['SE'],3),
                 'R2':round(r['R2'],3), 'Cluster':CLUSTER_NAMES[cl],
                 'Hit': '✓' if hit else '✗'})

df_oos = pd.DataFrame(rows)
print(df_oos.to_string(index=False))
df_oos.to_csv('../data/oos_fit_results.csv', index=False)

n_hit = sum(1 for r in rows if r['Hit']=='✓')
print(f"\nCluster hit rate: {n_hit}/{len(rows)}  ({100*n_hit/len(rows):.0f}%)")


In [ ]:
# OOS prediction accuracy
all_pred, all_obs, all_cols = [], [], []
for nm, r in oos_fits.items():
    pred = r['lnC'] + r['alpha']*r['x']
    all_pred.extend(pred); all_obs.extend(r['y'])
    all_cols.extend([COLORS.get(r['strategy'],'gray')]*r['n'])

all_pred, all_obs = np.array(all_pred), np.array(all_obs)
r2  = 1 - ((all_obs-all_pred)**2).sum() / ((all_obs-all_obs.mean())**2).sum()
mae = float(np.abs(all_obs-all_pred).mean())
print(f"OOS  R² = {r2:.3f}   MAE = {mae:.3f}  (log-space, {len(all_obs)} obs)")

# Benchmark comparison
all_aum_is = np.concatenate([r['aum'] for r in fits.values()])
all_hc_is  = np.concatenate([r['hc']  for r in fits.values()])
pool = ols_powerlaw(all_aum_is, all_hc_is)

all_aum_oos = np.concatenate([r['aum'] for r in oos_fits.values()])
all_hc_oos  = np.concatenate([r['hc']  for r in oos_fits.values()])
x_oos_all = np.log(all_aum_oos); y_oos_all = np.log(all_hc_oos)

mae_intercept = float(np.abs(y_oos_all - y_oos_all.mean()).mean())
pred_pool     = pool['lnC'] + pool['alpha']*x_oos_all
mae_pool      = float(np.abs(y_oos_all - pred_pool).mean())

print(f"\nBenchmark comparison (OOS MAE in log-space):")
print(f"  Intercept-only  : {mae_intercept:.3f}")
print(f"  Pooled power law: {mae_pool:.3f}")
print(f"  Fund-specific   : {mae:.3f}  ({(1-mae/mae_intercept)*100:.0f}% reduction vs intercept-only)")


In [ ]:
# OOS scatter: predicted vs observed
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.scatter(all_obs, all_pred, c=all_cols, s=22, alpha=0.72, edgecolors='none')
mn, mx = min(all_obs)-.1, max(all_obs)+.1
ax.plot([mn,mx],[mn,mx],'k--',lw=1.0)
ax.fill_between([mn,mx],[mn-.3,mx-.3],[mn+.3,mx+.3],alpha=0.07,color='k')
ax.set_xlabel('ln Nₛ  (observed)'); ax.set_ylabel('ln Nₛ  (predicted)')
ax.set_title(f'OOS: predicted vs observed  (R² = {r2:.3f}, MAE = {mae:.3f})')

# Alpha bands
ax = axes[1]
oos_sorted = sorted(oos_fits.items(), key=lambda x: x[1]['alpha'])
for i, (nm, r) in enumerate(oos_sorted):
    se = r['SE'] if r['SE'] > 0 else 0.1
    ax.errorbar(r['alpha'], i, xerr=1.96*se,
                fmt=MARKERS.get(r['strategy'],'o'),
                color=COLORS.get(r['strategy'],'gray'), capsize=3, ms=5)

# Shade cluster alpha bands
band_lo = [min(fits[nm]['alpha'] for nm,l in zip(fund_names,labels) if l==k) for k in range(3)]
band_hi = [max(fits[nm]['alpha'] for nm,l in zip(fund_names,labels) if l==k) for k in range(3)]
for k in range(3):
    ax.axvspan(band_lo[k], band_hi[k], alpha=0.10, color=CLUSTER_COLORS[k],
               label=CLUSTER_NAMES[k])

short_oos = [nm.split()[0] for nm,_ in oos_sorted]
ax.set_yticks(range(len(oos_sorted))); ax.set_yticklabels(short_oos, fontsize=6.5)
ax.set_xlabel('α̂  (OOS fund  ±1.96 SE)')
ax.set_title('OOS alpha bands')
ax.legend(fontsize=7, loc='lower right')

plt.tight_layout()
plt.savefig('../figures/oos_validation.png', dpi=150, bbox_inches='tight')
plt.show()


## 3.7  Combined in-sample + OOS log-log scatter

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6.5))

# IS fits (grey)
for nm, r in fits.items():
    ax.scatter(r['aum'], r['hc'], color='lightgray', s=28, zorder=2,
               edgecolors='k', lw=0.2, alpha=0.9)
    xl = np.linspace(r['aum'].min(), r['aum'].max(), 60)
    ax.plot(xl, r['C']*xl**r['alpha'], color='#aaaaaa', lw=0.7, ls='--', alpha=0.5)

# Cluster alpha corridors
aum_rng = np.linspace(2, 230, 300)
for k in range(3):
    alo = min(fits[nm]['alpha'] for nm,l in zip(fund_names,labels) if l==k)
    ahi = max(fits[nm]['alpha'] for nm,l in zip(fund_names,labels) if l==k)
    Cm  = np.median([fits[nm]['C'] for nm,l in zip(fund_names,labels) if l==k])
    ax.fill_between(aum_rng, Cm*aum_rng**alo, Cm*aum_rng**ahi,
                    color=CLUSTER_COLORS[k], alpha=0.07)

# OOS points
for nm, r in oos_fits.items():
    strat = r['strategy']
    ax.scatter(r['aum'], r['hc'],
               color=COLORS.get(strat,'gray'), marker=MARKERS.get(strat,'o'),
               s=38, zorder=3, alpha=0.85, edgecolors='k', lw=0.2)

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('AUM  (USD billion)'); ax.set_ylabel('Headcount')
ax.set_title('Combined in-sample + OOS log-log scatter')

patches = ([mpatches.Patch(color='lightgray', label='In-sample fits')] +
           [mpatches.Patch(color=COLORS[s], label=META.loc[s,'strategy_label'])
            for s in ['Q','P','M','F','A'] if s in COLORS])
ax.legend(handles=patches, fontsize=7.5, ncol=2, loc='upper left')
plt.tight_layout()
plt.savefig('../figures/combined_loglog.png', dpi=150, bbox_inches='tight')
plt.show()


## 3.8  Survivorship-bias correction — paper vs full-universe cluster fractions

In [ ]:
# Corrected fractions from paper Table (Tab. corrected_fracs)
# Full universe: 40 funds including 8 defunct

bias_data = {
    'year'  : [2010, 2013, 2015, 2018, 2020, 2022, 2024],
    'n_paper': [6,    7,    7,    8,    9,   10,   10],
    'n_full' : [6,   10,   19,   24,   29,   31,   30],
    # paper sample fractions
    'p_alg' : [0.33, 0.43, 0.71, 0.50, 0.44, 0.40, 0.40],
    'p_hyb' : [0.67, 0.57, 0.29, 0.38, 0.44, 0.40, 0.40],
    'p_pod' : [0.00, 0.00, 0.00, 0.12, 0.11, 0.20, 0.20],
    # full-universe fractions
    'f_alg' : [0.33, 0.60, 0.68, 0.62, 0.62, 0.58, 0.60],
    'f_hyb' : [0.67, 0.40, 0.32, 0.29, 0.31, 0.29, 0.30],
    'f_pod' : [0.00, 0.00, 0.00, 0.08, 0.07, 0.13, 0.10],
}
df_bias = pd.DataFrame(bias_data)
print(df_bias.to_string(index=False))
df_bias.to_csv('../data/survivorship_correction.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
yrs = df_bias['year'].values

for ax, (pk, fk, title) in zip(axes, [
        ('p_alg','f_alg','Algorithmic Scale  (Cluster I)'),
        ('p_hyb','f_hyb','Hybrid Platform  (Cluster II)'),
        ('p_pod','f_pod','Pod-Shop Linear  (Cluster III)')]):
    ax.plot(yrs, df_bias[pk], 'o--', color='tomato',   lw=1.5, label='Paper sample  (11 funds)')
    ax.plot(yrs, df_bias[fk], 's-',  color='steelblue',lw=1.8, label='Full universe  (40 funds)')
    ax.fill_between(yrs, df_bias[pk], df_bias[fk], alpha=0.12,
                    color='gray', label='Bias  (full − paper)')
    ax.set_ylim(0, 1); ax.set_xlabel('Year'); ax.set_ylabel('Cluster fraction')
    ax.set_title(title); ax.legend(fontsize=7)

plt.suptitle('Survivorship-bias correction — cluster fractions', y=1.01, fontsize=11)
plt.tight_layout()
plt.savefig('../figures/survivorship_correction.png', dpi=150, bbox_inches='tight')
plt.show()
